In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()

if not (ROOT / "src" / "model_data.py").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from model_data import load_features, cv_folds, TARGET

print("Repository:", ROOT)
print("Shared modelling framework loaded successfully")

Repository: C:\Users\gadha\sakura_analytics
Shared modelling framework loaded successfully


In [2]:
feb_df = load_features("feb1", source="csv")
mar_df = load_features("mar1", source="csv")

print("February 1:", feb_df.shape)
print("March 1:", mar_df.shape)

display(feb_df.head())

Loaded model_features_feb1 from data/model/ CSV: 4,365 rows
Loaded model_features_mar1 from data/model/ CSV: 4,365 rows
February 1: (4365, 33)
March 1: (4365, 36)


,station_id,location,prefecture,region,latitude,longitude,elevation_m,bloom_year,relocations_before_season,tmean_oct,...,sunshine_jan_h,sunshine_jan_missing_days,full_bloom_date,full_bloom_doy,split,fold1,fold2,fold3,fold4,fold5
0,412,Sapporo,Hokkaido,Hokkaido,43.060000,141.328333,17.4,1964,0,10.57,...,99.92,1.0,1964-05-07,128,development,train,train,train,train,train
1,423,Muroran,Hokkaido,Hokkaido,42.311667,140.975000,39.9,1964,0,12.19,...,100.10,0.0,1964-05-12,133,development,train,train,train,train,train
2,428,Esashi,Hokkaido,Hokkaido,41.866667,140.123333,3.7,1964,0,12.58,...,37.80,0.0,1964-05-08,129,development,train,train,train,train,train
3,430,Hakodate,Hokkaido,Hokkaido,41.816667,140.753333,35.0,1964,0,11.24,...,101.00,0.0,1964-05-10,131,development,train,train,train,train,train
4,433,Kutchan,Hokkaido,Hokkaido,42.900000,140.756667,176.1,1964,0,8.48,...,73.90,0.0,1964-05-13,134,development,train,train,train,train,train


In [3]:
keys = ["station_id", "bloom_year"]

feb_check = feb_df[keys + [TARGET]].rename(
    columns={TARGET: "target_feb"}
)

mar_check = mar_df[keys + [TARGET]].rename(
    columns={TARGET: "target_mar"}
)

comparison = feb_check.merge(
    mar_check,
    on=keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("Station-year matching:")
display(comparison["_merge"].value_counts())

matched = comparison[comparison["_merge"] == "both"]

print(
    "Target mismatches:",
    (
        matched["target_feb"] != matched["target_mar"]
    ).sum()
)

print("Missing February targets:", feb_df[TARGET].isna().sum())
print("Missing March targets:", mar_df[TARGET].isna().sum())

Station-year matching:


_merge
both          4365
left_only        0
right_only       0
Name: count, dtype: int64

Target mismatches: 0
Missing February targets: 0
Missing March targets: 0


In [4]:
print("Unique stations:", feb_df["station_id"].nunique())

print(
    "Year range:",
    feb_df["bloom_year"].min(),
    "-",
    feb_df["bloom_year"].max()
)

print("\nDevelopment observations:")
print((feb_df["bloom_year"] <= 2014).sum())

print("\nReserved test observations:")
print(feb_df["bloom_year"].between(2015, 2025).sum())

print("\nDuplicate station-years:")
print(feb_df.duplicated(keys).sum())

Unique stations: 82
Year range: 1964 - 2025

Development observations:
3837

Reserved test observations:
528

Duplicate station-years:
0


In [5]:
feb_features = set(feb_df.columns)
mar_features = set(mar_df.columns)

print("Features shared by both datasets:")
print(sorted(feb_features & mar_features))

print("\nFebruary-only features:")
print(sorted(feb_features - mar_features))

print("\nMarch-only features:")
print(sorted(mar_features - feb_features))

Features shared by both datasets:
['bloom_year', 'elevation_m', 'fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'full_bloom_date', 'full_bloom_doy', 'latitude', 'location', 'longitude', 'prefecture', 'region', 'relocations_before_season', 'split', 'station_id', 'tmax_jan', 'tmean_dec', 'tmean_jan', 'tmean_nov', 'tmean_oct', 'tmin_jan']

February-only features:
['chill_days_0_7c_nov_jan', 'cum_tmax_jan', 'cum_tmean_jan', 'frost_days_jan', 'gdd5_jan', 'precip_jan_missing_days', 'precip_jan_mm', 'sunshine_jan_h', 'sunshine_jan_missing_days', 'tmin_lowest_nov_jan']

March-only features:
['chill_days_0_7c_nov_feb', 'cum_tmax_feb', 'cum_tmean_feb', 'frost_days_jan_feb', 'gdd5_feb', 'precip_feb_missing_days', 'precip_feb_mm', 'sunshine_feb_h', 'sunshine_feb_missing_days', 'tmax_feb', 'tmean_feb', 'tmin_feb', 'tmin_lowest_nov_feb']


In [6]:
missing_summary = pd.DataFrame({
    "February_missing_pct": (
        feb_df.isna().mean() * 100
    ),
    "March_missing_pct": (
        mar_df.isna().mean() * 100
    )
}).fillna(0)

display(
    missing_summary[
        (missing_summary > 0).any(axis=1)
    ].sort_values(
        "February_missing_pct",
        ascending=False
    )
)

,February_missing_pct,March_missing_pct
fold1,21.878580,21.878580
fold2,18.969072,18.969072
fold3,16.540664,16.540664
fold4,14.295533,14.295533
fold5,12.096220,12.096220


In [7]:
expected_folds = [
    ((1964, 2004), (2005, 2006)),
    ((1964, 2006), (2007, 2008)),
    ((1964, 2008), (2009, 2010)),
    ((1964, 2010), (2011, 2012)),
    ((1964, 2012), (2013, 2014))
]

fold_summary = []

for i, (train_period, val_period) in enumerate(
    expected_folds, start=1
):
    train = feb_df[
        feb_df["bloom_year"].between(*train_period)
    ]

    val = feb_df[
        feb_df["bloom_year"].between(*val_period)
    ]

    fold_summary.append({
        "Fold": i,
        "Training": f"{train_period[0]}-{train_period[1]}",
        "Validation": f"{val_period[0]}-{val_period[1]}",
        "Train_N": len(train),
        "Validation_N": len(val)
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary)

,Fold,Training,Validation,Train_N,Validation_N
0,1,1964-2004,2005-2006,3262,148
1,2,1964-2006,2007-2008,3410,127
2,3,1964-2008,2009-2010,3537,106
3,4,1964-2010,2011-2012,3643,98
4,5,1964-2012,2013-2014,3741,96


In [8]:
verification_summary = pd.DataFrame({
    "Check": [
        "February observations",
        "March observations",
        "Unique stations",
        "Matching station-year records",
        "Target mismatches",
        "Duplicate station-years",
        "Development observations",
        "Reserved test observations"
    ],
    "Result": [
        len(feb_df),
        len(mar_df),
        feb_df["station_id"].nunique(),
        len(matched),
        (matched["target_feb"] != matched["target_mar"]).sum(),
        feb_df.duplicated(keys).sum(),
        (feb_df["bloom_year"] <= 2014).sum(),
        feb_df["bloom_year"].between(2015, 2025).sum()
    ]
})

display(verification_summary)

OUTPUT = ROOT / "outputs" / "baseline"
OUTPUT.mkdir(parents=True, exist_ok=True)

verification_summary.to_csv(
    OUTPUT / "shared_data_verification.csv",
    index=False
)

print("Verification summary saved.")

,Check,Result
0,February observations,4365
1,March observations,4365
2,Unique stations,82
3,Matching station-year records,4365
4,Target mismatches,0
5,Duplicate station-years,0
6,Development observations,3837
7,Reserved test observations,528


Verification summary saved.


In [9]:

from IPython.display import display, Markdown

summary = """
# Key Findings — Shared Data Verification

## 1. Dataset Overview

The modelling datasets were prepared using the team's
standardized Sakura Yosoku framework.

| Component | Result |
|---|---|
| Cherry species | Somei-yoshino |
| Observation period | 1964–2025 |
| Observation stations | 82 |
| February 1 dataset | 4,365 records |
| March 1 dataset | 4,365 records |
| Development observations | 3,837 |
| Reserved test observations | 528 |
| Prediction target | Full-bloom day of year (DOY) |

---

## 2. Key Findings

### Finding 1 — Consistent Forecast Datasets

Both February 1 and March 1 datasets contain identical
station-year combinations and target values.

- **Matching station-year records:** 4,365
- **Target mismatches:** 0
- **Duplicate station-year records:** 0
- **Missing full-bloom targets:** 0

This supports fair comparisons between the two
forecast issue dates.

### Finding 2 — Forecast-Specific Feature Availability

The February 1 dataset includes weather features
calculated through January.

The March 1 dataset additionally includes February
weather features, such as mean temperature,
maximum temperature and accumulated warmth.

No obvious post-cutoff weather features were identified
from the feature names.

### Finding 3 — Shared Validation Framework

The team uses five expanding-window validation folds:

| Fold | Training | Validation |
|---|---|---|
| 1 | 1964–2004 | 2005–2006 |
| 2 | 1964–2006 | 2007–2008 |
| 3 | 1964–2008 | 2009–2010 |
| 4 | 1964–2010 | 2011–2012 |
| 5 | 1964–2012 | 2013–2014 |

The final 2015–2025 test period is reserved
for independent evaluation.

### Finding 4 — Data Quality Considerations

Earlier source-data validation identified:

- Incomplete daily temperature observations at Onahama.
- Limited historical bloom coverage for Yakushima.
- Uneven observation histories across stations.

These findings were retained as modelling considerations
rather than automatically removing valid observations.

---

## 3. Limitations

- Feature names alone cannot establish that preprocessing
  is completely free from data leakage.
- Weather imputation must be checked against forecast cutoffs.
- Station relocation and historical coverage differences
  may influence modelling performance.
- The shared modelling datasets are preprocessed inputs;
  this notebook does not independently reproduce the ETL.

---

## 4. Conclusion

**The standardized February 1 and March 1 datasets
passed the initial structural consistency checks.**

Both datasets contain the same 4,365 station-year
observations, enabling comparable model development.

The shared datasets and validation framework were
adopted for subsequent baseline modelling.
"""

display(Markdown(summary))



# Key Findings — Shared Data Verification

## 1. Dataset Overview

The modelling datasets were prepared using the team's
standardized Sakura Yosoku framework.

| Component | Result |
|---|---|
| Cherry species | Somei-yoshino |
| Observation period | 1964–2025 |
| Observation stations | 82 |
| February 1 dataset | 4,365 records |
| March 1 dataset | 4,365 records |
| Development observations | 3,837 |
| Reserved test observations | 528 |
| Prediction target | Full-bloom day of year (DOY) |

---

## 2. Key Findings

### Finding 1 — Consistent Forecast Datasets

Both February 1 and March 1 datasets contain identical
station-year combinations and target values.

- **Matching station-year records:** 4,365
- **Target mismatches:** 0
- **Duplicate station-year records:** 0
- **Missing full-bloom targets:** 0

This supports fair comparisons between the two
forecast issue dates.

### Finding 2 — Forecast-Specific Feature Availability

The February 1 dataset includes weather features
calculated through January.

The March 1 dataset additionally includes February
weather features, such as mean temperature,
maximum temperature and accumulated warmth.

No obvious post-cutoff weather features were identified
from the feature names.

### Finding 3 — Shared Validation Framework

The team uses five expanding-window validation folds:

| Fold | Training | Validation |
|---|---|---|
| 1 | 1964–2004 | 2005–2006 |
| 2 | 1964–2006 | 2007–2008 |
| 3 | 1964–2008 | 2009–2010 |
| 4 | 1964–2010 | 2011–2012 |
| 5 | 1964–2012 | 2013–2014 |

The final 2015–2025 test period is reserved
for independent evaluation.

### Finding 4 — Data Quality Considerations

Earlier source-data validation identified:

- Incomplete daily temperature observations at Onahama.
- Limited historical bloom coverage for Yakushima.
- Uneven observation histories across stations.

These findings were retained as modelling considerations
rather than automatically removing valid observations.

---

## 3. Limitations

- Feature names alone cannot establish that preprocessing
  is completely free from data leakage.
- Weather imputation must be checked against forecast cutoffs.
- Station relocation and historical coverage differences
  may influence modelling performance.
- The shared modelling datasets are preprocessed inputs;
  this notebook does not independently reproduce the ETL.

---

## 4. Conclusion

**The standardized February 1 and March 1 datasets
passed the initial structural consistency checks.**

Both datasets contain the same 4,365 station-year
observations, enabling comparable model development.

The shared datasets and validation framework were
adopted for subsequent baseline modelling.
